# Group 2: train the Gaussian Splat (Colab)

Only training needs a GPU, so this notebook does only that. The rest runs on a laptop:

```
laptop:  frames -> check_images.py -> run_sfm.py -> dataset.zip        (~2 min)
Colab:   dataset.zip -> train.py -> splat.ply                            (this notebook, ~10 min)
laptop:  splat.ply -> voxel/ply_to_schem.py -> object.schem              (~10 s)
```

**Runtime -> Change runtime type -> T4 GPU** first. Then run the cells in order.

## 1. Set up (about 2-3 min, nothing compiles)

Colab's own Python changes every few months, and gsplat only ships ready-built GPU code for Python 3.10 +
PyTorch 2.4. So training runs in its own Python 3.10 environment (`/content/gs`), installed from an exact,
tested list of packages (`group2/colab_requirements.txt`). Colab's own Python is left alone.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
REPO, BRANCH = "https://github.com/arjunch1001/acmAI-3d-reconstruction.git", "group2-pipeline"
!rm -rf /content/repo && git clone -q --depth 1 --branch $BRANCH $REPO /content/repo
%cd /content/repo
!pip install -q uv
!rm -rf /content/gs && uv venv -q --python 3.10 /content/gs
INDEXES = "--extra-index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://docs.gsplat.studio/whl/pt24cu124 --index-strategy unsafe-best-match"
!uv pip install -q --python /content/gs/bin/python -r group2/colab_requirements.txt $INDEXES
assert _exit_code == 0, "Install failed (see above)."
PY = "/content/gs/bin/python"
!$PY group2/gpu_check.py
assert _exit_code == 0, "gsplat can't run on this GPU (see above)."


## 2. Upload `dataset.zip`

Made on the laptop by `python group2/run_sfm.py <frames> <work_folder>` (it prints the zip's path).
Check `python group2/sfm_report.py <work_folder> <frames>` says **OK to train** first.

In [ ]:
import os, zipfile
from google.colab import files
uploaded = os.path.abspath(next(iter(files.upload())))
!rm -rf /content/dataset
zipfile.ZipFile(uploaded).extractall("/content/dataset")
assert os.path.exists("/content/dataset/sparse/0/cameras.bin"), "Not a run_sfm.py dataset.zip (no sparse/0)."
print(len(os.listdir("/content/dataset/images")), "images")

## 3. Train

Prints loss, PSNR, Gaussian count, speed and time left every 100 steps. 3,000 steps should take a few minutes
on a T4. Add `--holdout 8` to also score views the model never trained on.

In [ ]:
!$PY group2/train.py /content/dataset /content/run --steps 3000
assert _exit_code == 0, "train.py failed (see above)." 

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
m = pd.read_csv("/content/run/metrics.csv")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, col, title in zip(ax, ["loss", "psnr", "gaussians"], ["loss", "train PSNR (dB)", "Gaussians"]):
    a.plot(m.step, m[col]); a.set_title(title)
plt.show()
display(Image(sorted(glob.glob("/content/run/previews/*.png"))[-1]))  # left: photo, right: render
print(open("/content/run/summary.json").read())

## 4. Download

`splat.ply` goes to Group 3 (or drag it into superspl.at/editor), and into the Minecraft step on the laptop:
`python voxel/ply_to_schem.py splat.ply object.schem --cameras <work_folder>/dataset/sparse/0`

In [ ]:
!cd /content/run && zip -q -j /content/outputs.zip splat.ply summary.json metrics.csv
files.download("/content/outputs.zip")